In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
ls /kaggle/input/competitions/tpu-getting-started

sample_submission.csv    tfrecords-jpeg-224x224/  tfrecords-jpeg-512x512/
tfrecords-jpeg-192x192/  tfrecords-jpeg-331x331/


In [3]:
# Config
IMAGE_SIZE = [224, 224]

In [4]:
import os
import io
import glob
import struct
from multiprocessing import Pool, cpu_count
from PIL import Image
from tqdm import tqdm

from tensorflow.core.example.example_pb2 import Example


# ---- TFRecord Reader ----
def read_tfrecord(file_path):
    with open(file_path, "rb") as f:
        while True:
            length_bytes = f.read(8)
            if not length_bytes:
                break

            length = struct.unpack("<Q", length_bytes)[0]
            f.read(4)  # crc
            data = f.read(length)
            f.read(4)  # crc

            yield data


# ---- Parser ----
def parse_example(serialized):
    example = Example()
    example.ParseFromString(serialized)

    features = example.features.feature

    image = features["image"].bytes_list.value[0]
    label = features["class"].int64_list.value[0]
    img_id = features["id"].bytes_list.value[0].decode()

    return image, label, img_id


# ---- Worker ----
def process_file(file_path):
    local_count = 0

    for record in read_tfrecord(file_path):
        img_bytes, label, img_id = parse_example(record)

        class_dir = os.path.join(OUTPUT_PATH, str(label))
        os.makedirs(class_dir, exist_ok=True)

        img = Image.open(io.BytesIO(img_bytes))
        img.save(os.path.join(class_dir, f"{img_id}.jpg"))

        local_count += 1

    return local_count

2026-05-03 06:31:20.002603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777789880.208164      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777789880.272130      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777789880.759874      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777789880.759910      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777789880.759913      57 computation_placer.cc:177] computation placer alr

In [5]:
BASE_PATH = "/kaggle/input/competitions/tpu-getting-started"

In [6]:
OUTPUT_PATH = "/kaggle/working/train_images"
os.makedirs(OUTPUT_PATH, exist_ok=True)
INPUT_PATH = BASE_PATH + "/tfrecords-jpeg-224x224/train"
# ---- Run Parallel ----
files_train = glob.glob(INPUT_PATH + "/*.tfrec")

num_workers = min(8, cpu_count())  # Kaggle usually 2–4 CPUs

with Pool(num_workers) as p:
    results = list(tqdm(p.imap(process_file, files_train), total=len(files_train)))

print(f"Total train images processed: {sum(results)}")

100%|██████████| 16/16 [00:05<00:00,  2.70it/s]

Total train images processed: 12753


In [7]:
OUTPUT_PATH = "/kaggle/working/val_images"
os.makedirs(OUTPUT_PATH, exist_ok=True)
INPUT_PATH = BASE_PATH + "/tfrecords-jpeg-224x224/val"
# ---- Run Parallel ----
files_val = glob.glob(INPUT_PATH + "/*.tfrec")

num_workers = min(8, cpu_count())  # Kaggle usually 2–4 CPUs

with Pool(num_workers) as p:
    results = list(tqdm(p.imap(process_file, files_val), total=len(files_val)))

print(f"Total validation images processed: {sum(results)}")

100%|██████████| 16/16 [00:01<00:00,  9.87it/s]

Total validation images processed: 3712


In [8]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = ImageFolder(
    root="/kaggle/working/train_images",
    transform=transform
)

val_dataset = ImageFolder(
    root="/kaggle/working/val_images",
    transform=transform
)

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64, # optimise it 64-128
    shuffle=True,
    num_workers=2, # optimise it 2-4
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64, 
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [18]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

images, labels = next(iter(val_loader))
print(images.shape, labels.shape)

torch.Size([64, 3, 224, 224]) torch.Size([64])
torch.Size([64, 3, 224, 224]) torch.Size([64])


In [19]:
import timm
import torch
import torch.nn as nn

NUM_CLASSES = 104  # flower classes

def build_model(model_name="convnext_tiny", pretrained=True):
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=NUM_CLASSES
    )
    return model

model_convnext_tiny = build_model("convnext_tiny")

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_convnext_tiny = model_convnext_tiny.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model_convnext_tiny.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [21]:
scaler = torch.cuda.amp.GradScaler()

In [14]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [22]:
EPOCHS = 16

best_acc = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model_convnext_tiny, train_loader, optimizer, scaler)
    val_acc = validate(model_convnext_tiny, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_convnext_tiny.state_dict(), "best_model_convnext_tiny.pth")

100%|██████████| 200/200 [01:07<00:00,  2.96it/s]


Epoch 1 | Loss: 1.7502 | Val Acc: 0.8370


100%|██████████| 200/200 [00:57<00:00,  3.50it/s]


Epoch 2 | Loss: 1.0737 | Val Acc: 0.8860


100%|██████████| 200/200 [00:57<00:00,  3.51it/s]


...

Epoch 23 | Loss: 0.8355 | Val Acc: 0.8863


100%|██████████| 200/200 [00:56<00:00,  3.52it/s]


Epoch 24 | Loss: 0.8023 | Val Acc: 0.9087


100%|██████████| 200/200 [00:56<00:00,  3.52it/s]


Epoch 25 | Loss: 0.7888 | Val Acc: 0.9221


In [23]:
model_convnext_tiny = build_model("convnext_tiny", pretrained=False)

In [26]:
model_convnext_tiny.load_state_dict(torch.load("/kaggle/working/best_model_convnext_tiny.pth", map_location=device))

model_convnext_tiny.to(device)

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)


In [27]:
import cv2
import numpy as np
import os

TEST_OUTPUT_PATH = "/kaggle/working/test_images"
os.makedirs(TEST_OUTPUT_PATH, exist_ok=True)

from tensorflow.core.example.example_pb2 import Example

def parse_test_example(serialized):
    example = Example()
    example.ParseFromString(serialized)

    features = example.features.feature

    image = features["image"].bytes_list.value[0]
    img_id = features["id"].bytes_list.value[0].decode()

    return image, img_id

def process_test_file(file_path):
    count = 0

    for record in read_tfrecord(file_path):
        img_bytes, img_id = parse_test_example(record)

        # fast decode
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        cv2.imwrite(os.path.join(TEST_OUTPUT_PATH, f"{img_id}.jpg"), img)

        count += 1

    return count

In [28]:
TEST_INPUT_PATH = BASE_PATH + "/tfrecords-jpeg-224x224/test"

files_test = glob.glob(TEST_INPUT_PATH + "/*.tfrec")

num_workers = min(4, cpu_count())  # safer on Kaggle

with Pool(num_workers) as p:
    results = list(tqdm(p.imap(process_test_file, files_test), total=len(files_test)))

print(f"Total test images processed: {sum(results)}")

100%|██████████| 16/16 [00:03<00:00,  4.78it/s]

Total test images processed: 7382


In [31]:
from torch.utils.data import Dataset
from PIL import Image
import os

class TestDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.paths = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".jpg")
        ])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        img_id = os.path.basename(path).replace(".jpg", "")
        return img, img_id

In [32]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

test_dataset = TestDataset(
    "/kaggle/working/test_images",
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [33]:
ids = []
preds = []

model_convnext_tiny.eval()

with torch.no_grad():
    for images, img_ids in test_loader:
        images = images.to(device)

        outputs = model_convnext_tiny(images)
        pred = outputs.argmax(dim=1).cpu().numpy()

        ids.extend(img_ids)
        preds.extend(pred)

In [34]:
import pandas as pd

df = pd.DataFrame({
    "id": ids,
    "label": preds
})

df.to_csv("submission.csv", index=False)

In [46]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),  # flowers → vertical flip is useful
    transforms.RandomRotation(20),
    
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    transforms.RandomErasing(p=0.25)
])

train_dataset = ImageFolder(
    root="/kaggle/working/train_images",
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64, # optimise it 64-128
    shuffle=True,
    num_workers=1, # optimise it 2-4
    pin_memory=True,
    persistent_workers=True
)

In [47]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = len(train_dataset.classes)

model_convnext_tiny = timm.create_model(
    "convnext_tiny",
    pretrained=False,   # important when loading weights
    num_classes=NUM_CLASSES
)

model_convnext_tiny.load_state_dict(torch.load(
    "/kaggle/working/best_model_convnext_tiny.pth",
    map_location=device
))

model_convnext_tiny.to(device)

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)


In [48]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model_convnext_tiny.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

scaler = torch.cuda.amp.GradScaler()

In [75]:
val_loader = DataLoader(
    val_dataset,
    batch_size=64, 
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [51]:
EPOCHS = 10

best_acc = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model_convnext_tiny, train_loader, optimizer, scaler)
    val_acc = validate(model_convnext_tiny, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_convnext_tiny.state_dict(), "best_model_convnext_tiny_aug.pth")

100%|██████████| 200/200 [01:46<00:00,  1.88it/s]


Epoch 1 | Loss: 1.1132 | Val Acc: 0.8664


100%|██████████| 200/200 [01:46<00:00,  1.88it/s]


Epoch 2 | Loss: 1.0629 | Val Acc: 0.8710


100%|██████████| 200/200 [01:46<00:00,  1.89it/s]


...

Epoch 8 | Loss: 0.8182 | Val Acc: 0.9362


100%|██████████| 200/200 [01:46<00:00,  1.87it/s]


Epoch 9 | Loss: 0.8125 | Val Acc: 0.9378


100%|██████████| 200/200 [01:51<00:00,  1.80it/s]


Epoch 10 | Loss: 0.8057 | Val Acc: 0.9399


In [52]:
for epoch in range(5):
    train_loss = train_one_epoch(model_convnext_tiny, train_loader, optimizer, scaler)
    val_acc = validate(model_convnext_tiny, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_convnext_tiny.state_dict(), "best_model_convnext_tiny_aug.pth")

100%|██████████| 200/200 [01:45<00:00,  1.89it/s]


Epoch 1 | Loss: 0.8065 | Val Acc: 0.9399


100%|██████████| 200/200 [01:45<00:00,  1.89it/s]


Epoch 2 | Loss: 0.8054 | Val Acc: 0.9405


100%|██████████| 200/200 [01:45<00:00,  1.89it/s]


Epoch 3 | Loss: 0.8065 | Val Acc: 0.9391


100%|██████████| 200/200 [01:45<00:00,  1.90it/s]


Epoch 4 | Loss: 0.8171 | Val Acc: 0.9321


100%|██████████| 200/200 [01:45<00:00,  1.90it/s]


Epoch 5 | Loss: 0.8473 | Val Acc: 0.9230


In [53]:
model_efficient_b0 = build_model("efficientnet_b0")

device = "cuda" if torch.cuda.is_available() else "cpu"
model_efficient_b0 = model_efficient_b0.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model_efficient_b0.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

scaler = torch.cuda.amp.GradScaler()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [74]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),  # flowers → vertical flip is useful
    transforms.RandomRotation(15),
    
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    transforms.RandomErasing(p=0.25)
])

train_dataset = ImageFolder(
    root="/kaggle/working/train_images",
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64, # optimise it 64-128
    shuffle=True,
    num_workers=2, # optimise it 2-4
    pin_memory=True,
    persistent_workers=True
)

In [56]:
# freeze everything
for param in model_efficient_b0.parameters():
    param.requires_grad = False

# unfreeze classifier safely
for name, param in model_efficient_b0.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

optimizer = torch.optim.AdamW(
    model_efficient_b0.classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

HEAD_EPOCHS = 2

for epoch in range(HEAD_EPOCHS):
    train_loss = train_one_epoch(model_efficient_b0, train_loader, optimizer, scaler)
    val_acc = validate(model_efficient_b0, val_loader)

    print(f"[HEAD] Epoch {epoch+1} | Loss: {train_loss:.4f} | Val: {val_acc:.4f}")

100%|██████████| 200/200 [01:48<00:00,  1.85it/s]


[HEAD] Epoch 1 | Loss: 3.8669 | Val: 0.3297


100%|██████████| 200/200 [01:45<00:00,  1.90it/s]


[HEAD] Epoch 2 | Loss: 2.9312 | Val: 0.4989


In [57]:
EPOCHS = 15

for param in model_efficient_b0.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    model_efficient_b0.parameters(),
    lr=1e-4,   # 🔥 lower LR for fine-tuning
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [58]:
best_acc = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model_efficient_b0, train_loader, optimizer, scaler)
    val_acc = validate(model_efficient_b0, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_efficient_b0.state_dict(), "best_model_efficient_b0_aug.pth")

100%|██████████| 200/200 [01:56<00:00,  1.72it/s]


Epoch 1 | Loss: 2.0023 | Val Acc: 0.7823


100%|██████████| 200/200 [01:44<00:00,  1.91it/s]


Epoch 2 | Loss: 1.5220 | Val Acc: 0.8400


100%|██████████| 200/200 [01:46<00:00,  1.89it/s]


...

Epoch 13 | Loss: 0.9741 | Val Acc: 0.9025


100%|██████████| 200/200 [01:47<00:00,  1.86it/s]


Epoch 14 | Loss: 0.9737 | Val Acc: 0.9036


100%|██████████| 200/200 [01:46<00:00,  1.89it/s]


Epoch 15 | Loss: 0.9739 | Val Acc: 0.9030


In [60]:
for epoch in range(10):
    train_loss = train_one_epoch(model_efficient_b0, train_loader, optimizer, scaler)
    val_acc = validate(model_efficient_b0, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_efficient_b0.state_dict(), "best_model_efficient_b0_aug.pth")

100%|██████████| 200/200 [01:45<00:00,  1.90it/s]


Epoch 1 | Loss: 0.9750 | Val Acc: 0.9030


100%|██████████| 200/200 [01:45<00:00,  1.89it/s]


Epoch 2 | Loss: 0.9681 | Val Acc: 0.9041


100%|██████████| 200/200 [01:45<00:00,  1.89it/s]


...

Epoch 8 | Loss: 0.9570 | Val Acc: 0.9030


100%|██████████| 200/200 [01:46<00:00,  1.88it/s]


Epoch 9 | Loss: 0.9521 | Val Acc: 0.9071


100%|██████████| 200/200 [01:46<00:00,  1.87it/s]


Epoch 10 | Loss: 0.9426 | Val Acc: 0.9044


# MixUp and CutMix

In [61]:
import numpy as np
import torch

def mixup(images, labels, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(images.size(0)).to(images.device)

    mixed_images = lam * images + (1 - lam) * images[index]
    labels_a, labels_b = labels, labels[index]

    return mixed_images, labels_a, labels_b, lam

In [62]:
def cutmix(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size, _, H, W = images.size()
    index = torch.randperm(batch_size).to(images.device)

    # bounding box
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    # apply cutmix
    images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]

    # adjust lambda
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))

    labels_a, labels_b = labels, labels[index]

    return images, labels_a, labels_b, lam

In [63]:
import random

def mixup_cutmix(images, labels, mixup_prob=0.7):
    if random.random() < mixup_prob:
        return mixup(images, labels)
    else:
        return cutmix(images, labels)

In [68]:
def train_one_epoch_mixup_cut_mix(model, loader, optimizer, scaler, model_ema=None):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        # 🔥 Mixup/CutMix auto-switch
        images, labels_a, labels_b, lam = mixup_cutmix(images, labels)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)

            loss = (
                lam * criterion(outputs, labels_a) +
                (1 - lam) * criterion(outputs, labels_b)
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if model_ema:
            model_ema.update(model)

        total_loss += loss.item()

    return total_loss / len(loader)

In [66]:
model_efficient_b0 = build_model("efficientnet_b0")

from timm.utils import ModelEma

model_ema = ModelEma(
    model_efficient_b0,
    decay=0.999,    # smoothing factor
    device=device   # keep on same device for speed
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_efficient_b0 = model_efficient_b0.to(device)

# freeze everything
for param in model_efficient_b0.parameters():
    param.requires_grad = False

# unfreeze classifier safely
for name, param in model_efficient_b0.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

optimizer = torch.optim.AdamW(
    model_efficient_b0.classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

HEAD_EPOCHS = 5

for epoch in range(HEAD_EPOCHS):
    train_loss = train_one_epoch_mixup_cut_mix(model_efficient_b0, train_loader, optimizer, scaler, model_ema)
    val_acc = validate(model_efficient_b0, val_loader)

    print(f"[HEAD] Epoch {epoch+1} | Loss: {train_loss:.4f} | Val: {val_acc:.4f}")

[HEAD] Epoch 1 | Loss: 4.1161 | Val: 0.2546
[HEAD] Epoch 2 | Loss: 3.4713 | Val: 0.4327
[HEAD] Epoch 3 | Loss: 3.1117 | Val: 0.5267
[HEAD] Epoch 4 | Loss: 2.9029 | Val: 0.5663
[HEAD] Epoch 5 | Loss: 2.7996 | Val: 0.5854


In [80]:
EPOCHS = 50

for param in model_efficient_b0.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    model_efficient_b0.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

warmup_epochs = 3

scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs),
        CosineAnnealingLR(optimizer, T_max=EPOCHS - warmup_epochs)
    ],
    milestones=[warmup_epochs]
)

scaler = torch.cuda.amp.GradScaler()

best_acc = 0
best_acc_ema = 0

patience = 10
no_improve_epochs = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch_mixup_cut_mix(
        model_efficient_b0,
        train_loader,
        optimizer,
        scaler,
        model_ema
    )

    val_acc = validate(model_efficient_b0, val_loader)
    val_acc_ema = validate(model_ema.ema, val_loader)  # ✅ fix

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val: {val_acc:.4f}")
    print(f"Epoch {epoch+1} | EMA Val: {val_acc_ema:.4f}")

    improved = False

    # ---- Base model save ----
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_efficient_b0.state_dict(), "best_model_efficient_b0.pth")

    # ---- EMA model save (MAIN metric) ----
    if val_acc_ema > best_acc_ema:
        best_acc_ema = val_acc_ema
        torch.save(model_ema.ema.state_dict(), "best_model_efficient_b0_ema.pth")
        improved = True

    # ---- Early stopping logic ----
    if improved:
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    if no_improve_epochs >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

100%|██████████| 200/200 [01:00<00:00,  3.33it/s]


Epoch 1 | Loss: 2.2235 | Val: 0.7961
Epoch 1 | EMA Val: 0.6651


100%|██████████| 200/200 [01:00<00:00,  3.29it/s]


Epoch 2 | Loss: 2.2029 | Val: 0.8314
Epoch 2 | EMA Val: 0.7241


100%|██████████| 200/200 [01:01<00:00,  3.28it/s]


...

Epoch 35 | Loss: 1.4227 | Val: 0.9297
Epoch 35 | EMA Val: 0.9318


100%|██████████| 200/200 [01:00<00:00,  3.30it/s]


Epoch 36 | Loss: 1.4336 | Val: 0.9310
Epoch 36 | EMA Val: 0.9318


100%|██████████| 200/200 [00:59<00:00,  3.34it/s]


Epoch 37 | Loss: 1.4085 | Val: 0.9305
Epoch 37 | EMA Val: 0.9321
Early stopping triggered at epoch 37
